# Unit Cell Fabrication Pipeline - Multi-Prompt Testing

Complete pipeline: Prompt → Geometry Agent → Endpoint Generator → GWL Serialization

This notebook tests all 4 prompts from `prompt.txt` and generates complete outputs for each.

## Setup

In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Add current directory to path to import local modules
SCRIPT_DIR = Path.cwd()
sys.path.insert(0, str(SCRIPT_DIR))

# Import the geometry agent
from Unit_Cell_Geometry_Agent import (
    client,
    SYSTEM_PROMPT,
    UNIT_CELL_SCHEMA,
    identify_unit_cell,
    save_output,
    build_graph,
    AgentState
)

# Import endpoint generator
from endpoint_generator import (
    generate_endpoint_json,
    load_print_parameters
)

# Import GWL serializer
from gwl_serializer import (
    generate_gwl_files,
    load_gwl_parameters,
    generate_master_gwl
)

print("✓ Modules imported successfully")

## Load All Test Prompts

In [ ]:
# Read all prompts from prompt.txt
prompt_file = SCRIPT_DIR / "prompt.txt"
with open(prompt_file, 'r', encoding='utf-8') as f:
    content = f.read()

# Split by empty lines and filter out empty entries
raw_prompts = [p.strip() for p in content.split('\n\n') if p.strip()]

# Parse prompts with their labels
PROMPTS = []
for prompt in raw_prompts:
    # Extract label (e.g., "Base-", "Undergrad-", etc.)
    if '-' in prompt[:20]:  # Label should be near start
        label, text = prompt.split('-', 1)
        PROMPTS.append({
            'label': label.strip(),
            'text': text.strip()
        })

print(f"Loaded {len(PROMPTS)} test prompts:")
for i, p in enumerate(PROMPTS):
    print(f"  [{i+1}] {p['label']}")
    print(f"      {p['text'][:80]}..." if len(p['text']) > 80 else f"      {p['text']}")
    print()

## Process All Prompts

This cell runs the complete pipeline for each prompt:
1. Geometry Agent → Unit Cell JSON
2. Endpoint Generator → Scan Endpoints
3. GWL Serializer → Nanoscribe Files
4. Visualization → Layer Preview

In [ ]:
# Load print parameters once
print_params = load_print_parameters(SCRIPT_DIR / "PrintParameters.txt")
gwl_params = load_gwl_parameters(SCRIPT_DIR / "PrintParameters.txt")

print(f"Print Parameters:")
print(f"  Slice distance: {print_params['slice_distance_um']} µm")
print(f"  Hatch distance: {print_params['hatch_distance_um']} µm")
print(f"  Voxel XY: {print_params['voxel_xy_um']} µm\n")

# Storage for results
results = []

# Process each prompt
for idx, prompt_data in enumerate(PROMPTS):
    label = prompt_data['label']
    prompt_text = prompt_data['text']
    
    print("="*80)
    print(f"PROCESSING [{idx+1}/{len(PROMPTS)}]: {label}")
    print("="*80)
    print(f"Prompt: {prompt_text[:100]}..." if len(prompt_text) > 100 else f"Prompt: {prompt_text}")
    print()
    
    # Step 1: Run Geometry Agent
    print(f"[1/4] Running Geometry Agent...")
    initial_state = {
        "prompt": prompt_text,
        "category": "",
        "result": {},
        "output_path": "",
        "token_usage": {}
    }
    
    state_after_identify = identify_unit_cell(initial_state)
    unit_cell_data = state_after_identify['result']
    token_usage = state_after_identify['token_usage']
    
    print(f"  ✓ Unit cell: {unit_cell_data['job_name']}")
    print(f"  ✓ Components: {len(unit_cell_data['unit_cell']['components'])}")
    print(f"  ✓ Array: {unit_cell_data['global_info']['repetitions']['x']}×{unit_cell_data['global_info']['repetitions']['y']}")
    print(f"  ✓ Tokens: {token_usage['total_tokens']}\n")
    
    # Step 2: Generate Endpoints
    print(f"[2/4] Generating Endpoints...")
    endpoint_data = generate_endpoint_json(unit_cell_data, print_params)
    
    total_layers = len(endpoint_data['layers'])
    total_segments = sum(len(layer['segments']) for layer in endpoint_data['layers'])
    z_min = endpoint_data['layers'][0]['z_um']
    z_max = endpoint_data['layers'][-1]['z_um']
    
    print(f"  ✓ Layers: {total_layers}")
    print(f"  ✓ Segments: {total_segments:,}")
    print(f"  ✓ Z range: {z_min:.2f} to {z_max:.2f} µm\n")
    
    # Step 3: Generate GWL Files
    print(f"[3/4] Generating GWL Files...")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    gwl_output_dir = SCRIPT_DIR / "Outputs" / f"{label}_GWL_{unit_cell_data['job_name']}_{timestamp}"
    gwl_output_dir.mkdir(parents=True, exist_ok=True)
    
    gwl_files = generate_gwl_files(endpoint_data, gwl_params, gwl_output_dir)
    master_file = gwl_output_dir / f"{unit_cell_data['job_name']}_master.gwl"
    generate_master_gwl(gwl_files, gwl_params, master_file)
    
    print(f"  ✓ Generated {len(gwl_files)} layer files + master")
    print(f"  ✓ Output: {gwl_output_dir.name}\n")
    
    # Step 4: Save JSONs
    print(f"[4/4] Saving Results...")
    json_output_dir = SCRIPT_DIR / "Outputs" / f"{label}_JSON_{unit_cell_data['job_name']}_{timestamp}"
    json_output_dir.mkdir(parents=True, exist_ok=True)
    
    unit_cell_file = json_output_dir / "unit_cell.json"
    with open(unit_cell_file, 'w') as f:
        json.dump(unit_cell_data, f, indent=2)
    
    endpoints_file = json_output_dir / "endpoints.json"
    with open(endpoints_file, 'w') as f:
        json.dump(endpoint_data, f, indent=2)
    
    print(f"  ✓ Saved unit_cell.json ({unit_cell_file.stat().st_size:,} bytes)")
    print(f"  ✓ Saved endpoints.json ({endpoints_file.stat().st_size:,} bytes)")
    print(f"  ✓ Output: {json_output_dir.name}\n")
    
    # Store results for summary
    results.append({
        'label': label,
        'prompt': prompt_text,
        'unit_cell_data': unit_cell_data,
        'endpoint_data': endpoint_data,
        'total_layers': total_layers,
        'total_segments': total_segments,
        'z_range': (z_min, z_max),
        'tokens': token_usage['total_tokens'],
        'gwl_dir': gwl_output_dir,
        'json_dir': json_output_dir
    })
    
    print(f"✓ COMPLETED: {label}\n")

print("="*80)
print(f"ALL {len(PROMPTS)} PROMPTS PROCESSED SUCCESSFULLY")
print("="*80)

## Summary of All Results

In [ ]:
# Print summary table
print("\n" + "="*100)
print("SUMMARY OF ALL TEST CASES")
print("="*100)
print(f"{'Label':<12} {'Job Name':<25} {'Layers':<8} {'Segments':<12} {'Z Range (µm)':<20} {'Tokens':<8}")
print("-"*100)

for r in results:
    z_min, z_max = r['z_range']
    print(f"{r['label']:<12} {r['unit_cell_data']['job_name']:<25} {r['total_layers']:<8} {r['total_segments']:<12,} {f'{z_min:.1f} - {z_max:.1f}':<20} {r['tokens']:<8}")

print("="*100)

# Print output directories
print("\nOutput Directories:")
for r in results:
    print(f"\n{r['label']}:")
    print(f"  GWL:  {r['gwl_dir']}")
    print(f"  JSON: {r['json_dir']}")

## Visualizations - Layer 0 for All Prompts

In [ ]:
# Create 2x2 grid of visualizations
fig, axes = plt.subplots(2, 2, figsize=(18, 18))
axes = axes.flatten()

for idx, r in enumerate(results):
    ax = axes[idx]
    
    # Get layer 0 data
    layer = r['endpoint_data']['layers'][0]
    segments = layer['segments']
    z_height = layer['z_um']
    
    # Plot segments
    for seg in segments:
        x = [seg['start'][0], seg['end'][0]]
        y = [seg['start'][1], seg['end'][1]]
        ax.plot(x, y, 'b-', linewidth=0.5, alpha=0.7)
    
    # Formatting
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlabel('X (µm)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Y (µm)', fontsize=11, fontweight='bold')
    ax.set_title(f"{r['label']} - Layer 0\nZ = {z_height:.2f} µm | {len(segments):,} segments",
                 fontsize=12, fontweight='bold')
    
    # Info box
    info_text = f"""Array: {r['unit_cell_data']['global_info']['repetitions']['x']}×{r['unit_cell_data']['global_info']['repetitions']['y']}
Components: {len(r['unit_cell_data']['unit_cell']['components'])}
Total Layers: {r['total_layers']}
Total Segments: {r['total_segments']:,}"""
    
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes,
            fontsize=9, verticalalignment='top', family='monospace',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8, edgecolor='navy'))

plt.tight_layout()
plt.show()

print("\n✓ Layer 0 visualizations complete for all prompts")

## Individual Prompt Deep Dive

Select a specific prompt to examine in detail

In [ ]:
# Select which result to examine (0=Base, 1=Undergrad, 2=Grad, 3=Postdoc)
SELECTED_IDX = 2  # Change this to examine different prompts

selected = results[SELECTED_IDX]
print(f"Selected: {selected['label']}")
print(f"Job: {selected['unit_cell_data']['job_name']}")
print(f"Prompt: {selected['prompt'][:150]}..." if len(selected['prompt']) > 150 else f"Prompt: {selected['prompt']}")
print(f"\nComponents:")
for i, comp in enumerate(selected['unit_cell_data']['unit_cell']['components']):
    z_center = comp['center'][2]
    height = comp['dimensions'].get('height_um', 0)
    z_min = z_center - height/2
    z_max = z_center + height/2
    print(f"  [{i}] {comp['type']}")
    print(f"      Dimensions: {comp['dimensions']}")
    print(f"      Z range: [{z_min:.1f}, {z_max:.1f}] µm")

## Layer Browser for Selected Prompt

In [ ]:
# Select layer to visualize
LAYER_INDEX = 0  # Change this to view different layers

selected_endpoint = selected['endpoint_data']
total_layers = len(selected_endpoint['layers'])

print(f"Available layers: 0 to {total_layers-1}")
print(f"\nFirst 10 layer heights (µm):")
for i in range(min(10, total_layers)):
    layer = selected_endpoint['layers'][i]
    print(f"  Layer {i:3d}: Z = {layer['z_um']:6.2f} µm ({len(layer['segments']):6,} segments)")

# Validate
if LAYER_INDEX >= total_layers:
    print(f"\nWarning: Layer {LAYER_INDEX} doesn't exist. Using layer {total_layers-1}")
    LAYER_INDEX = total_layers - 1

selected_layer = selected_endpoint['layers'][LAYER_INDEX]
z_height = selected_layer['z_um']
segments = selected_layer['segments']

print(f"\n→ Visualizing Layer {LAYER_INDEX}")
print(f"  Z height: {z_height} µm")
print(f"  Scan segments: {len(segments):,}")

# Visualization
fig, ax = plt.subplots(figsize=(14, 14))

for seg in segments:
    x = [seg['start'][0], seg['end'][0]]
    y = [seg['start'][1], seg['end'][1]]
    ax.plot(x, y, 'b-', linewidth=0.5, alpha=0.7)

ax.set_aspect('equal')
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xlabel('X (µm)', fontsize=13, fontweight='bold')
ax.set_ylabel('Y (µm)', fontsize=13, fontweight='bold')
ax.set_title(f'{selected["label"]} - Layer {LAYER_INDEX} - Z = {z_height:.2f} µm\n{len(segments):,} segments',
             fontsize=15, fontweight='bold', pad=20)

info_text = f"""Job: {selected['unit_cell_data']['job_name']}
Array: {selected['unit_cell_data']['global_info']['repetitions']['x']}×{selected['unit_cell_data']['global_info']['repetitions']['y']}
Spacing: {selected['unit_cell_data']['global_info']['spacing']['x_um']} µm
Components: {len(selected['unit_cell_data']['unit_cell']['components'])}
Total Layers: {selected['total_layers']}
Total Segments: {selected['total_segments']:,}"""

ax.text(0.02, 0.98, info_text, transform=ax.transAxes,
        fontsize=11, verticalalignment='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.9, edgecolor='navy', linewidth=2))

plt.tight_layout()
plt.show()

print(f"\n✓ Visualization complete")